# Notebook 06 — S&P 500 Market Regime Detection

This notebook identifies historical market regimes using observable price and volatility behavior.

**Input**
`data/raw/sp500_1950_present.csv`

**Regime methods**
1. Rule-based trend/volatility classification
2. Hidden Markov Model (HMM) using `hmmlearn`

The regime analysis uses information available at each date and is intended for research. It does not use future returns as regime features.

The notebook does not modify the raw master dataset.


## 1. Imports

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import StandardScaler

try:
    from hmmlearn.hmm import GaussianHMM
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "hmmlearn"])
    from hmmlearn.hmm import GaussianHMM

warnings.filterwarnings("ignore")

print("Imports loaded successfully.")


ModuleNotFoundError: No module named 'hmmlearn'

## 2. Configuration and Paths

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/quant-trading-research"),
    ]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break

MASTER_PATH = PROJECT_ROOT / "data" / "raw" / "sp500_1950_present.csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
REPORT_DIR = PROJECT_ROOT / "reports" / "generated"

for path in [INTERIM_DIR, FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EXPECTED_COLUMNS = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj.Close",
    "Volume",
]

print(f"Master dataset: {MASTER_PATH}")


## 3. Load and Validate the Master Dataset

In [ ]:
if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"Master dataset not found: {MASTER_PATH}. "
        "Run the earlier notebooks first."
    )

df = pd.read_csv(MASTER_PATH, low_memory=False)

if list(df.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Unexpected schema. Expected {EXPECTED_COLUMNS}; "
        f"received {list(df.columns)}"
    )

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

for column in EXPECTED_COLUMNS[1:]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = (
    df.sort_values("Date")
    .drop_duplicates("Date")
    .reset_index(drop=True)
)

if df["Date"].isna().any():
    raise ValueError("Invalid dates detected.")

if df[["Open", "High", "Low", "Close", "Adj.Close", "Volume"]].isna().any().any():
    raise ValueError("Missing OHLCV values detected.")

if not df["Date"].is_unique:
    raise ValueError("Duplicate dates detected.")

print(f"Rows: {len(df):,}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")


## 4. Construct Regime Features

All regime features are backward-looking.

Features:
- daily return
- rolling realized volatility
- short/long moving-average relationship
- moving-average slope
- rolling drawdown
- rolling return
- normalized trading range


In [ ]:
df["return_1d"] = df["Close"].pct_change()

df["volatility_21d"] = (
    df["return_1d"]
    .rolling(21)
    .std()
    * np.sqrt(252)
)

df["volatility_63d"] = (
    df["return_1d"]
    .rolling(63)
    .std()
    * np.sqrt(252)
)

df["sma_50"] = df["Close"].rolling(50).mean()
df["sma_200"] = df["Close"].rolling(200).mean()

df["trend_spread"] = (
    df["sma_50"] / df["sma_200"] - 1
)

df["sma_50_slope"] = (
    df["sma_50"] / df["sma_50"].shift(50) - 1
)

df["return_63d"] = (
    df["Close"] / df["Close"].shift(63) - 1
)

df["range_pct"] = (
    (df["High"] - df["Low"]) / df["Close"]
)

df["wealth_index"] = (
    1 + df["return_1d"].fillna(0)
).cumprod()

df["running_peak"] = df["wealth_index"].cummax()

df["drawdown"] = (
    df["wealth_index"] / df["running_peak"] - 1
)

feature_columns = [
    "return_1d",
    "volatility_21d",
    "volatility_63d",
    "trend_spread",
    "sma_50_slope",
    "return_63d",
    "range_pct",
    "drawdown",
]

display(df[feature_columns].describe().T)


## 5. Rule-Based Trend Regime

A transparent rule-based classifier provides an interpretable baseline.

Definitions:
- Bull: price trend positive and 50-day SMA above 200-day SMA
- Bear: price trend negative and 50-day SMA below 200-day SMA
- Neutral: otherwise

This is deliberately simple and will serve as a benchmark for the HMM.


In [ ]:
bull_condition = (
    (df["trend_spread"] > 0)
    & (df["return_63d"] > 0)
    & (df["sma_50_slope"] > 0)
)

bear_condition = (
    (df["trend_spread"] < 0)
    & (df["return_63d"] < 0)
    & (df["sma_50_slope"] < 0)
)

df["trend_regime"] = np.select(
    [bull_condition, bear_condition],
    ["Bull", "Bear"],
    default="Neutral",
)

trend_regime_summary = (
    df["trend_regime"]
    .value_counts()
    .rename_axis("regime")
    .reset_index(name="observations")
)

trend_regime_summary["percentage"] = (
    trend_regime_summary["observations"] / len(df)
)

display(trend_regime_summary)

trend_regime_summary.to_csv(
    TABLE_DIR / "sp500_rule_based_trend_regimes.csv",
    index=False
)


## 6. Volatility Regime

Volatility is divided into three historical quantile groups:

- Low
- Medium
- High

The quantile thresholds are calculated from the historical sample for descriptive regime analysis.


In [ ]:
volatility_valid = df["volatility_21d"].dropna()

volatility_q33 = volatility_valid.quantile(1 / 3)
volatility_q67 = volatility_valid.quantile(2 / 3)

df["volatility_regime"] = np.select(
    [
        df["volatility_21d"] <= volatility_q33,
        df["volatility_21d"] >= volatility_q67,
    ],
    [
        "Low Volatility",
        "High Volatility",
    ],
    default="Medium Volatility",
)

volatility_regime_summary = (
    df["volatility_regime"]
    .value_counts()
    .rename_axis("regime")
    .reset_index(name="observations")
)

volatility_regime_summary["percentage"] = (
    volatility_regime_summary["observations"] / len(df)
)

display(volatility_regime_summary)

volatility_regime_summary.to_csv(
    TABLE_DIR / "sp500_volatility_regimes.csv",
    index=False
)


## 7. Combined Trend + Volatility Regime

Combining trend and volatility creates a more useful descriptive market-state classification.


In [ ]:
df["combined_regime"] = (
    df["trend_regime"].astype(str)
    + " / "
    + df["volatility_regime"].astype(str)
)

combined_regime_summary = (
    df["combined_regime"]
    .value_counts()
    .rename_axis("regime")
    .reset_index(name="observations")
)

combined_regime_summary["percentage"] = (
    combined_regime_summary["observations"] / len(df)
)

display(combined_regime_summary)

combined_regime_summary.to_csv(
    TABLE_DIR / "sp500_combined_regime_summary.csv",
    index=False
)


## 8. Rule-Based Regime Visualization

In [ ]:
regime_codes = {
    "Bear": -1,
    "Neutral": 0,
    "Bull": 1,
}

regime_numeric = df["trend_regime"].map(regime_codes)

fig = plt.figure(figsize=(14, 5))
plt.plot(df["Date"], regime_numeric)
plt.yticks([-1, 0, 1], ["Bear", "Neutral", "Bull"])
plt.title("Rule-Based S&P 500 Trend Regimes")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_rule_based_trend_regimes.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 9. Regime-Conditional Return Statistics

This evaluates how historical returns and volatility behaved within each rule-based trend regime.

This is descriptive and does not use future information to assign the regime.


In [ ]:
regime_return_summary = (
    df.groupby("trend_regime")
    .agg(
        observations=("return_1d", "count"),
        mean_daily_return=("return_1d", "mean"),
        median_daily_return=("return_1d", "median"),
        daily_volatility=("return_1d", "std"),
        mean_21d_volatility=("volatility_21d", "mean"),
        mean_63d_return=("return_63d", "mean"),
        mean_drawdown=("drawdown", "mean"),
    )
    .reset_index()
)

regime_return_summary["annualized_return_approx"] = (
    regime_return_summary["mean_daily_return"] * 252
)

regime_return_summary["annualized_volatility"] = (
    regime_return_summary["daily_volatility"] * np.sqrt(252)
)

display(regime_return_summary)

regime_return_summary.to_csv(
    TABLE_DIR / "sp500_trend_regime_return_statistics.csv",
    index=False
)


## 10. Prepare HMM Features

The HMM uses standardized observable market features:

- daily return
- 21-day realized volatility
- trend spread
- 63-day return
- normalized trading range

Future returns are not included.


In [ ]:
hmm_features = [
    "return_1d",
    "volatility_21d",
    "trend_spread",
    "return_63d",
    "range_pct",
]

hmm_data = df[
    ["Date"] + hmm_features
].dropna().copy()

print(f"HMM observations: {len(hmm_data):,}")
display(hmm_data.head())


## 11. Standardize HMM Features

In [ ]:
scaler = StandardScaler()

X_hmm = scaler.fit_transform(
    hmm_data[hmm_features]
)

print(f"HMM matrix shape: {X_hmm.shape}")
print(
    "Feature means after scaling:",
    np.round(X_hmm.mean(axis=0), 6)
)
print(
    "Feature std after scaling:",
    np.round(X_hmm.std(axis=0), 6)
)


## 12. Fit HMM Models with Different State Counts

Several state counts are evaluated.

The goal is not to select the model purely by likelihood; interpretability, stability, and regime characteristics are considered as well.


In [ ]:
hmm_results = []
hmm_models = {}

for n_states in [2, 3, 4]:
    model = GaussianHMM(
        n_components=n_states,
        covariance_type="full",
        n_iter=300,
        random_state=42,
        tol=1e-4,
        verbose=False,
    )

    model.fit(X_hmm)

    log_likelihood = model.score(X_hmm)

    n_features = X_hmm.shape[1]
    n_params = (
        n_states * (n_states - 1)
        + n_states * n_features
        + n_states * n_features
        + n_states * n_features * (n_features + 1) / 2
    )

    aic = -2 * log_likelihood + 2 * n_params
    bic = -2 * log_likelihood + np.log(len(X_hmm)) * n_params

    hmm_models[n_states] = model

    hmm_results.append({
        "states": n_states,
        "log_likelihood": log_likelihood,
        "AIC": aic,
        "BIC": bic,
        "converged": bool(model.monitor_.converged),
        "iterations": int(model.monitor_.iter),
    })

hmm_model_comparison = pd.DataFrame(hmm_results)

display(hmm_model_comparison)

hmm_model_comparison.to_csv(
    TABLE_DIR / "sp500_hmm_model_comparison.csv",
    index=False
)


## 13. Select HMM State Count

The lowest BIC is used as a statistical reference, but the final interpretation must also inspect whether the states are meaningful and sufficiently populated.


In [ ]:
best_bic_states = int(
    hmm_model_comparison.loc[
        hmm_model_comparison["BIC"].idxmin(),
        "states"
    ]
)

print(f"Lowest-BIC state count: {best_bic_states}")

hmm_model = hmm_models[best_bic_states]

hmm_data["hmm_state"] = hmm_model.predict(X_hmm)

state_counts = (
    hmm_data["hmm_state"]
    .value_counts()
    .sort_index()
    .rename_axis("state")
    .reset_index(name="observations")
)

state_counts["percentage"] = (
    state_counts["observations"] / len(hmm_data)
)

display(state_counts)


## 14. Profile HMM States

Each HMM state is profiled using the original, unscaled features.

This allows the states to be interpreted economically rather than by arbitrary state number.


In [ ]:
hmm_profile = (
    hmm_data
    .groupby("hmm_state")[hmm_features]
    .agg(["mean", "std", "median"])
)

display(hmm_profile)

hmm_profile.to_csv(
    TABLE_DIR / "sp500_hmm_state_profiles.csv"
)


## 15. Map HMM States Back to the Main Dataset

In [ ]:
state_mapping = hmm_data[["Date", "hmm_state"]].copy()

df = df.merge(
    state_mapping,
    on="Date",
    how="left",
    validate="one_to_one",
)

print(
    f"Rows with HMM state assigned: "
    f"{df['hmm_state'].notna().sum():,} / {len(df):,}"
)


## 16. HMM State Visualization

In [ ]:
fig = plt.figure(figsize=(14, 6))
plt.plot(
    df["Date"],
    df["Close"]
)

for state in sorted(df["hmm_state"].dropna().unique()):
    mask = df["hmm_state"] == state
    plt.scatter(
        df.loc[mask, "Date"],
        df.loc[mask, "Close"],
        s=4,
        label=f"State {int(state)}"
    )

plt.title("S&P 500 Price with HMM Market States")
plt.xlabel("Date")
plt.ylabel("Close")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_hmm_market_states.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 17. HMM State Return Statistics

In [ ]:
hmm_return_summary = (
    df.dropna(subset=["hmm_state"])
    .groupby("hmm_state")
    .agg(
        observations=("return_1d", "count"),
        mean_daily_return=("return_1d", "mean"),
        median_daily_return=("return_1d", "median"),
        daily_volatility=("return_1d", "std"),
        mean_21d_volatility=("volatility_21d", "mean"),
        mean_63d_return=("return_63d", "mean"),
        mean_drawdown=("drawdown", "mean"),
    )
    .reset_index()
)

hmm_return_summary["annualized_return_approx"] = (
    hmm_return_summary["mean_daily_return"] * 252
)

hmm_return_summary["annualized_volatility"] = (
    hmm_return_summary["daily_volatility"] * np.sqrt(252)
)

display(hmm_return_summary)

hmm_return_summary.to_csv(
    TABLE_DIR / "sp500_hmm_state_return_statistics.csv",
    index=False
)


## 18. HMM State Transition Matrix

In [ ]:
transition_matrix = pd.DataFrame(
    hmm_model.transmat_,
    index=[
        f"State {i}"
        for i in range(best_bic_states)
    ],
    columns=[
        f"State {i}"
        for i in range(best_bic_states)
    ],
)

display(transition_matrix)

transition_matrix.to_csv(
    TABLE_DIR / "sp500_hmm_transition_matrix.csv"
)


## 19. HMM State Persistence

In [ ]:
state_persistence = pd.DataFrame({
    "state": range(best_bic_states),
    "self_transition_probability": np.diag(
        hmm_model.transmat_
    ),
})

state_persistence["approx_expected_duration_days"] = (
    1 /
    (1 - state_persistence["self_transition_probability"])
)

display(state_persistence)

state_persistence.to_csv(
    TABLE_DIR / "sp500_hmm_state_persistence.csv",
    index=False
)


## 20. Compare Rule-Based and HMM Regimes

In [ ]:
comparison = df.dropna(
    subset=["hmm_state"]
).groupby(
    ["trend_regime", "hmm_state"]
).size().reset_index(
    name="observations"
)

comparison["percentage_within_trend_regime"] = (
    comparison["observations"] /
    comparison.groupby("trend_regime")["observations"].transform("sum")
)

display(comparison)

comparison.to_csv(
    TABLE_DIR / "sp500_rule_vs_hmm_regime_comparison.csv",
    index=False
)


## 21. Regime Duration Analysis

In [ ]:
def calculate_streak_lengths(series):
    values = series.dropna().reset_index(drop=True)
    if len(values) == 0:
        return pd.DataFrame(columns=["state", "duration"])

    groups = (values != values.shift()).cumsum()

    return (
        pd.DataFrame({
            "state": values,
            "group": groups,
        })
        .groupby(["group", "state"])
        .size()
        .reset_index(name="duration")
        .drop(columns="group")
    )

hmm_streaks = calculate_streak_lengths(
    df["hmm_state"]
)

hmm_duration_summary = (
    hmm_streaks
    .groupby("state")["duration"]
    .agg(
        observations="count",
        mean="mean",
        median="median",
        maximum="max",
    )
    .reset_index()
)

display(hmm_duration_summary)

hmm_duration_summary.to_csv(
    TABLE_DIR / "sp500_hmm_regime_duration_summary.csv",
    index=False
)


## 22. HMM Regime Timeline

In [ ]:
fig = plt.figure(figsize=(14, 5))
plt.step(
    df["Date"],
    df["hmm_state"],
    where="post"
)
plt.title("S&P 500 HMM Regime Timeline")
plt.xlabel("Date")
plt.ylabel("HMM State")
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_hmm_regime_timeline.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 23. Regime-Conditioned Forward Returns

This is an exploratory check of how future returns differed after observations classified into each regime.

The HMM state itself is inferred using the full historical sequence, so this analysis is **descriptive rather than out-of-sample predictive**. Formal leakage-controlled regime modeling belongs in later validation work.


In [ ]:
for horizon, periods in {
    "1d": 1,
    "5d": 5,
    "21d": 21,
    "63d": 63,
}.items():
    df[f"forward_return_{horizon}"] = (
        df["Close"].shift(-periods) / df["Close"] - 1
    )

hmm_forward_summary = (
    df.dropna(subset=["hmm_state"])
    .groupby("hmm_state")
    .agg(
        observations=("return_1d", "count"),
        forward_1d=("forward_return_1d", "mean"),
        forward_5d=("forward_return_5d", "mean"),
        forward_21d=("forward_return_21d", "mean"),
        forward_63d=("forward_return_63d", "mean"),
    )
    .reset_index()
)

display(hmm_forward_summary)

hmm_forward_summary.to_csv(
    TABLE_DIR / "sp500_hmm_forward_return_summary.csv",
    index=False
)


## 24. Save Regime Research Dataset

In [ ]:
regime_output = INTERIM_DIR / "sp500_regime_detection.parquet"

df.to_parquet(
    regime_output,
    index=False
)

print(f"Saved: {regime_output}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")


## 25. Regime Detection Report

In [ ]:
regime_report = {
    "dataset": {
        "rows": int(len(df)),
        "start": df["Date"].min().strftime("%Y-%m-%d"),
        "end": df["Date"].max().strftime("%Y-%m-%d"),
    },
    "rule_based": {
        "trend_regimes": trend_regime_summary.to_dict(
            orient="records"
        ),
        "volatility_regimes": volatility_regime_summary.to_dict(
            orient="records"
        ),
        "combined_regimes": combined_regime_summary.to_dict(
            orient="records"
        ),
    },
    "hmm": {
        "model_comparison": hmm_model_comparison.to_dict(
            orient="records"
        ),
        "selected_state_count": best_bic_states,
        "state_counts": state_counts.to_dict(
            orient="records"
        ),
        "transition_matrix": hmm_model.transmat_.tolist(),
        "state_persistence": state_persistence.to_dict(
            orient="records"
        ),
    },
    "methodological_note": (
        "HMM states are inferred from the full historical sequence. "
        "Their forward-return analysis is descriptive and should not be "
        "interpreted as leakage-controlled predictive performance."
    ),
}

report_path = REPORT_DIR / "sp500_regime_detection_report.json"

report_path.write_text(
    json.dumps(regime_report, indent=2, default=str),
    encoding="utf-8"
)

print(json.dumps(regime_report, indent=2, default=str))
print(f"\nSaved: {report_path}")


## 26. Final Master Dataset Integrity Check

Regime analysis must not modify the raw acquisition CSV.


In [ ]:
master_check = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

assert list(master_check.columns) == EXPECTED_COLUMNS
assert len(master_check) > 0

master_dates = pd.to_datetime(
    master_check["Date"],
    errors="coerce"
)

assert master_dates.notna().all()
assert master_dates.is_unique
assert master_dates.is_monotonic_increasing

print("Raw master dataset integrity after regime analysis: PASS")
print(f"Master rows: {len(master_check):,}")


# Notebook 06 Complete

Notebook 06 has completed market-regime research using:

### Rule-based regimes
- Trend regime
- Volatility regime
- Combined trend/volatility regime

### HMM regimes
- 2-state model
- 3-state model
- 4-state model
- AIC/BIC comparison
- State profiling
- Transition matrix
- State persistence
- Regime duration
- Rule-based vs HMM comparison

### Important methodological boundary

The HMM is an exploratory historical regime detector. Because standard HMM inference uses the complete sequence, its forward-return analysis is descriptive and is **not** a leakage-controlled prediction result.

Leakage-controlled model selection and walk-forward validation occur later.

**Next notebook:** Notebook 07 — Statistical Models.

Run Notebook 06 from top to bottom and verify its outputs before proceeding.
